In [3]:
import pandas as pd
import torch
data_folder = "/biodata/nyanovsky/datasets/dti/processed/v2/"
nodes = pd.read_csv(f"{data_folder}node_df.csv", index_col="node_id")
edges = pd.read_csv(f"{data_folder}edge_df_dedup.csv")

tensor_df = pd.read_csv(f"{data_folder}dti_tensor_df.csv", index_col=0)


full_dataset = torch.load(f"{data_folder}dti_full_dataset.pt")

In [4]:
nodes.head()

,ChG_deg,ChCh_deg,GG_deg,total_deg,node_type,louvain community (nodetype subgraph)
node_id,,,,,,
C155831,1,4,0,5,chem,141
G5243,41,0,17,58,gene,42
C24762158,9,2,0,11,chem,489
G213,85,0,3,88,gene,2
G506,6,0,3,9,gene,93


In [5]:
edges.head()

,src_id,trgt_id,edge_type,src_node_type,trgt_node_type,src_node_index,trgt_node_index
0,C155831,G5243,chg,chem,gene,0,1
1,C24762158,G213,chg,chem,gene,2,3
2,C24762158,G506,chg,chem,gene,2,4
3,C24762158,G563,chg,chem,gene,2,5
4,C24762158,G13884,chg,chem,gene,2,6


In [4]:
tensor_df.head()

,node_id,ChG_deg,ChCh_deg,GG_deg,total_deg,node_type,tensor_index
0,C155831,1,4,0,5,chem,0
2,C24762158,9,2,0,11,chem,1
12,C54692492,3,10,0,13,chem,2
16,C12968471,8,0,0,8,chem,3
25,C24836820,1,23,0,24,chem,4


In [29]:
feature_ids = []
with open(data_folder+"prot_features_ids.txt", "r") as f:
    for line in f:
        feature_ids.append(line.strip())

In [2]:
import sys
sys.path.append("..")
from models.training_utils import load_feature_dict
from data_class import GNNData

In [6]:
data = GNNData(node_path="/biodata/nyanovsky/datasets/dti/processed/v2/node_df.csv",
               edge_path="/biodata/nyanovsky/datasets/dti/processed/v2/edge_df_dedup.csv",
               node_index_col="node_id",
               src_edge_col="src_id",
               dst_edge_col="trgt_id",
               node_type_col="node_type",
               edge_type_col="edge_type",
               src_node_type_col="src_node_type",
               dst_node_type_col="trgt_node_type")

In [7]:
data.data

HeteroData(
  chem={ num_nodes=5854 },
  gene={ num_nodes=5805 },
  (chem, chg, gene)={ edge_index=[2, 38391] },
  (chem, chch, chem)={ edge_index=[2, 40000] },
  (gene, gg, gene)={ edge_index=[2, 43660] },
  (gene, rev_chg, chem)={ edge_index=[2, 38391] }
)

In [5]:
full_dataset

HeteroData(
  chem={
    num_nodes=5854,
    degree_chg=[5854],
  },
  gene={
    num_nodes=5805,
    degree_chg=[5805],
  },
  (chem, chg, gene)={ edge_index=[2, 38391] },
  (chem, chch, chem)={ edge_index=[2, 40000] },
  (gene, gg, gene)={ edge_index=[2, 43660] },
  (gene, chg, chem)={ edge_index=[2, 38391] }
)

In [8]:
gene_feature_dict = load_feature_dict(data_folder+"prot_features_64.txt", data_folder+"prot_features_ids.txt", 
                                                    tensor_df, "gene")

In [14]:
data.initialize_features(64, gene_feature_dict, inplace=True)

/usr/users/nyanovsky/tesis/gnn_interface/data_class.py:135: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data_object.data[nodetype].x[tensor_idxs] = nodetype_embs


In [10]:
data.edge_df.shape

(80221, 8)